# 1. SED modelling of a single debris belt

This notebook builds the spectral energy distribution (SED) of a star plus one optically thin dust belt with `artefact`, step by step:

1. choose the star, dust and belt parameters,
2. run the `RTModel` workflow,
3. inspect the grain temperatures and size distribution,
4. see how the SED responds to the belt and the dust.

Run it from the `notebooks/` directory. `artefact` must be importable (`pip install -e .` from the repository root) and `matplotlib` installed.

In [ ]:
import sys
from pathlib import Path

# The notebooks live in notebooks/, the helpers in example/
sys.path.insert(0, str(Path.cwd().parent / "example"))

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

In [ ]:
from artefact import RTModel
from example_utils import ASTROSIL, G2V_SPECTRUM, base_parameters, plot_sed, run_model

## Parameters

Parameters are a plain dictionary (or a text file read with `RTModel.get_parameters`, see `example/artefact_parameters.txt`). Units: au, pc, microns, L$_\odot$, R$_\odot$, g cm$^{-3}$ and Earth masses of dust.

`stype` selects the stellar model: `"blackbody"` (needs `tstar`, `rstar`), `"spectrum"` (an SVO-style model file), `"json"` (an SDB file) or `"phoenix"` (fit to photometry). Here we use the solar-type spectrum shipped in `data/stellar_models`.

In [ ]:
parameters = base_parameters(
    mdust=1e-3,        # Earth masses of dust with amin < a < amax
    dtype="onepl",     # single power-law belt: "onepl", "twopl", "gauss" or "arbit"
    rin=40.0,          # au
    rout=60.0,         # au
    alpha_out=-1.0,    # surface density ~ r**alpha_out
    q=-3.5,            # grain size distribution dN/da ~ a**q
)
parameters

## The workflow, step by step

`run_model` in `example_utils.py` wraps these calls; they are shown explicitly here because the order matters (each step uses the previous one's results).

In [ ]:
model = RTModel()
model.parameters.update(parameters)

model.make_star()                    # photosphere on the model wavelength grid
model.make_dust()                    # grain sizes, numbers and masses
model.read_optical_constants()       # n, k of the dust (astrosil.lnk)
model.make_disc()                    # annuli and relative surface density
model.calculate_surface_density()    # grains per annulus and size
model.calculate_qabs()               # Mie theory: Qabs and Qsca for every grain
model.calculate_dust_scatter()       # scattered starlight
model.calculate_dust_emission(mode="full", tolerance=1e-3)  # thermal emission
model.flam_to_fnu()                  # convert everything to mJy

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
plot_sed(model, ax=ax, title="Solar twin at 10 pc, belt at 40-60 au")
plt.show()

## Inside the model

`model.tdust` holds the temperature of every grain size at every radius, and `model.ng` the number of grains in each size bin.

In [ ]:
fig, (ax_t, ax_n) = plt.subplots(1, 2, figsize=(12, 4.5))

for index in (0, len(model.ag) // 2, len(model.ag) - 1):
    ax_t.plot(model.radii, model.tdust[:, index], label=f"a = {model.ag[index]:.3g} um")
ax_t.set_xlabel("Radius (au)")
ax_t.set_ylabel("Dust temperature (K)")
ax_t.legend()

ax_n.loglog(model.ag, model.ng)
ax_n.set_xlabel("Grain radius (um)")
ax_n.set_ylabel("Number of grains per size bin")
plt.show()

Small grains are hotter than large ones at the same radius because they absorb starlight efficiently but radiate poorly at long wavelengths. This is why a blackbody fit to the SED gives a different radius from the real belt.

## Blackbody temperatures versus the full calculation

`mode="bb"` assumes every grain is a blackbody at the equilibrium temperature. It is far quicker, which is useful in fitting loops, but ignores the grain-size dependence above.

In [ ]:
bb = run_model(parameters, emission_mode="bb")

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(model.sed_wave, model.sed_emit, label="full")
ax.loglog(bb.sed_wave, bb.sed_emit, "--", label="blackbody grains")
ax.set_ylim(model.sed_emit.max() * 1e-4, model.sed_emit.max() * 3)
ax.set_xlabel("Wavelength (um)")
ax.set_ylabel("Thermal emission (mJy)")
ax.legend()
plt.show()

## Changing the belt

Move the belt out and make it narrower or broader. Each model takes a few seconds.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for rin, rout in [(10, 30), (40, 60), (100, 140)]:
    m = run_model(base_parameters(rin=rin, rout=rout, mdust=1e-3), emission_mode="bb")
    ax.loglog(m.sed_wave, m.sed_emit + m.sed_scat, label=f"{rin}-{rout} au")
ax.set_ylim(1e-3, 1e2)
ax.set_xlabel("Wavelength (um)")
ax.set_ylabel("Disc flux (mJy)")
ax.legend(title="Belt")
plt.show()

## Changing the dust

The size distribution slope `q` and minimum grain size `amin` set how much emission comes from small, hot grains versus large, cold ones, which changes the SED shape and the sub-mm slope.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for q in (-3.2, -3.5, -3.9):
    m = run_model(base_parameters(q=q), emission_mode="bb")
    ax.loglog(m.sed_wave, m.sed_emit, label=f"q = {q}")
ax.set_ylim(1e-3, 1e2)
ax.set_xlabel("Wavelength (um)")
ax.set_ylabel("Thermal emission (mJy)")
ax.legend()
plt.show()

## Dust mass to belt mass

`calculate_disc_mass` extends the fitted dust size distribution up to planetesimals with the three-slope model of Krivov & Wyatt (2021).

In [ ]:
print(f"Dust mass   : {model.parameters['mdust']:.3g} M_earth")
print(f"Total mass  : {model.calculate_disc_mass():.3g} M_earth")